In [1]:

# ============================================================
# 1. Imports
# ============================================================
import os
import gc
import json
import time
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet50_Weights
from PIL import Image

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))


PyTorch version: 2.10.0+cu128
CUDA available : True
GPU            : Tesla T4


In [2]:

# ============================================================
# 2. Configuration
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 6

MODEL_NAME = "ResNet50_Frozen_224_NoAug"

# Requested output location
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = WORKING_ROOT / "working_outputs" / "model_d"
MODEL_ROOT = OUTPUT_ROOT / "models"
FIG_ROOT = OUTPUT_ROOT / "figures"

for d in [OUTPUT_ROOT, MODEL_ROOT, FIG_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print("Output root:", OUTPUT_ROOT)


Output root: /kaggle/working/working_outputs/model_d


In [3]:

# ============================================================
# 2. Configuration
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 6

MODEL_NAME = "ResNet50_Frozen_224_Aug"

# Requested output location
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = WORKING_ROOT / "working_outputs" / "model_d"
MODEL_ROOT = OUTPUT_ROOT / "models"
FIG_ROOT = OUTPUT_ROOT / "figures"

for d in [OUTPUT_ROOT, MODEL_ROOT, FIG_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print("Output root:", OUTPUT_ROOT)


Output root: /kaggle/working/working_outputs/model_d


In [4]:
# ============================================================
# CHAPTER 3 — Input
# ============================================================
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")

def find_all_files(root: Path, filename: str):
    return list(root.rglob(filename))

def pick_one(matches, label):
    if not matches:
        print(f"[DEBUG] No matches found for: {label}")
        return None
    print(f"[DEBUG] Matches for {label}:")
    for m in matches:
        print("   ", m)
    return matches[0]

# find files anywhere under /kaggle/input
x_train_path = pick_one(find_all_files(KAGGLE_INPUT, "X_train.csv"), "X_train.csv")
y_train_path = pick_one(find_all_files(KAGGLE_INPUT, "Y_train.csv"), "Y_train.csv")
x_test_path  = pick_one(find_all_files(KAGGLE_INPUT, "X_test.csv"),  "X_test.csv")

train_split_path = pick_one(find_all_files(KAGGLE_INPUT, "train_split.csv"), "train_split.csv")
val_split_path   = pick_one(find_all_files(KAGGLE_INPUT, "val_split.csv"),   "val_split.csv")

# image folders
image_train_dirs = list(KAGGLE_INPUT.rglob("image_train"))
image_test_dirs = list(KAGGLE_INPUT.rglob("image_test"))

print("[DEBUG] image_train dirs found:")
for p in image_train_dirs:
    print("   ", p)

print("[DEBUG] image_test dirs found:")
for p in image_test_dirs:
    print("   ", p)

if x_train_path is None or y_train_path is None or x_test_path is None:
    raise FileNotFoundError(
        "Missing one or more required CSV files under /kaggle/input. "
        "Please check the exact filenames printed above."
    )

if train_split_path is None or val_split_path is None:
    raise FileNotFoundError(
        "Could not find train_split.csv and/or val_split.csv under /kaggle/input."
    )

if not image_train_dirs:
    raise FileNotFoundError("Could not find image_train folder under /kaggle/input.")

if not image_test_dirs:
    raise FileNotFoundError("Could not find image_test folder under /kaggle/input.")

CSV_DATASET_DIR = x_train_path.parent
IMAGE_TRAIN_DIR = image_train_dirs[0]
IMAGE_TEST_DIR = image_test_dirs[0]

print("\n[INFO] Using paths:")
print("X_train.csv      :", x_train_path)
print("Y_train.csv      :", y_train_path)
print("X_test.csv       :", x_test_path)
print("train_split.csv  :", train_split_path)
print("val_split.csv    :", val_split_path)
print("IMAGE_TRAIN_DIR  :", IMAGE_TRAIN_DIR)
print("IMAGE_TEST_DIR   :", IMAGE_TEST_DIR)

[DEBUG] Matches for X_train.csv:
    /kaggle/input/datasets/arturillenseer/csv-files/X_train.csv
[DEBUG] Matches for Y_train.csv:
    /kaggle/input/datasets/arturillenseer/csv-files/Y_train.csv
[DEBUG] Matches for X_test.csv:
    /kaggle/input/datasets/arturillenseer/csv-files/X_test.csv
[DEBUG] Matches for train_split.csv:
    /kaggle/input/datasets/arturillenseer/csv-files/train_split.csv
[DEBUG] Matches for val_split.csv:
    /kaggle/input/datasets/arturillenseer/csv-files/val_split.csv
[DEBUG] image_train dirs found:
    /kaggle/input/datasets/arturillenseer/rakuten-product-images-ml/image_train
[DEBUG] image_test dirs found:
    /kaggle/input/datasets/arturillenseer/rakuten-product-images-ml/image_test

[INFO] Using paths:
X_train.csv      : /kaggle/input/datasets/arturillenseer/csv-files/X_train.csv
Y_train.csv      : /kaggle/input/datasets/arturillenseer/csv-files/Y_train.csv
X_test.csv       : /kaggle/input/datasets/arturillenseer/csv-files/X_test.csv
train_split.csv  : /kaggle

In [5]:
# ============================================================
# 4. Input files
# ============================================================
from pathlib import Path

# normalize variable names from chapter 3
X_TRAIN_PATH = Path(x_train_path)
Y_TRAIN_PATH = Path(y_train_path)
X_TEST_PATH = Path(x_test_path)
TRAIN_SPLIT_PATH = Path(train_split_path)
VAL_SPLIT_PATH = Path(val_split_path)

CSV_DATASET_DIR = X_TRAIN_PATH.parent

LABEL2ID_CANDIDATES = [
    CSV_DATASET_DIR / "label2id.json",
    CSV_DATASET_DIR / "label_mapping.json",
    CSV_DATASET_DIR / "cnn_256_class_indices.json",
]

for p in [X_TRAIN_PATH, Y_TRAIN_PATH, X_TEST_PATH, TRAIN_SPLIT_PATH, VAL_SPLIT_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Required file not found: {p}")

print("Using split files:")
print("-", TRAIN_SPLIT_PATH)
print("-", VAL_SPLIT_PATH)


Using split files:
- /kaggle/input/datasets/arturillenseer/csv-files/train_split.csv
- /kaggle/input/datasets/arturillenseer/csv-files/val_split.csv


In [6]:

# ============================================================
# 5. Load fixed splits and label mapping
# ============================================================
train_df = pd.read_csv(TRAIN_SPLIT_PATH)
val_df = pd.read_csv(VAL_SPLIT_PATH)

print("Train shape:", train_df.shape)
print("Val shape  :", val_df.shape)

label2id = None
for candidate in LABEL2ID_CANDIDATES:
    if candidate.exists():
        with open(candidate, "r", encoding="utf-8") as f:
            loaded = json.load(f)

        # Support both direct label->id mapping and nested structures
        if "label_to_index" in loaded:
            loaded = loaded["label_to_index"]

        label2id = {int(k): int(v) for k, v in loaded.items()}
        print("Loaded label mapping from:", candidate)
        break

if label2id is None:
    labels = sorted(train_df["prdtypecode"].unique())
    label2id = {int(label): i for i, label in enumerate(labels)}
    print("Built label mapping from train_split.csv")

id2label = {v: k for k, v in label2id.items()}

train_df["label_id"] = train_df["prdtypecode"].map(label2id)
val_df["label_id"] = val_df["prdtypecode"].map(label2id)

if train_df["label_id"].isna().any() or val_df["label_id"].isna().any():
    raise ValueError("Some labels in the split files could not be mapped to label IDs.")

train_df["label_id"] = train_df["label_id"].astype(int)
val_df["label_id"] = val_df["label_id"].astype(int)

NUM_CLASSES = len(label2id)
print("Number of classes:", NUM_CLASSES)


Train shape: (67932, 6)
Val shape  : (16984, 6)
Built label mapping from train_split.csv
Number of classes: 27


In [7]:

# ============================================================
# 6. Rebuild image paths from productid and imageid
# ============================================================
def make_image_path(row, split="train"):
    fname = f"image_{row['imageid']}_product_{row['productid']}.jpg"
    if split == "train":
        return str(IMAGE_TRAIN_DIR / fname)
    if IMAGE_TEST_DIR is None:
        raise FileNotFoundError("image_test directory is not available.")
    return str(IMAGE_TEST_DIR / fname)

train_df["image_path_local"] = train_df.apply(lambda row: make_image_path(row, split="train"), axis=1)
val_df["image_path_local"] = val_df.apply(lambda row: make_image_path(row, split="train"), axis=1)

train_exists = train_df["image_path_local"].apply(lambda p: Path(p).exists())
val_exists = val_df["image_path_local"].apply(lambda p: Path(p).exists())

print("Train image existence rate:", f"{train_exists.mean():.4f}")
print("Val image existence rate  :", f"{val_exists.mean():.4f}")

if not train_exists.all():
    missing = train_df.loc[~train_exists, ["productid", "imageid", "image_path_local"]].head(10)
    raise FileNotFoundError(f"Missing training images detected. Examples:\n{missing}")

if not val_exists.all():
    missing = val_df.loc[~val_exists, ["productid", "imageid", "image_path_local"]].head(10)
    raise FileNotFoundError(f"Missing validation images detected. Examples:\n{missing}")


Train image existence rate: 1.0000
Val image existence rate  : 1.0000


In [8]:
# ============================================================
# 7. Transforms, Dataset, DataLoaders
# ============================================================

from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        size=IMAGE_SIZE,
        scale=(0.85, 1.0)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=14),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


class RakutenImageDataset(Dataset):
    def __init__(self, df, transform=None, path_col="image_path_local", label_col="label_id"):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.path_col = path_col
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, self.path_col]
        label = int(self.df.loc[idx, self.label_col])

        image = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return image, label


train_dataset = RakutenImageDataset(train_df, transform=train_transform)
val_dataset = RakutenImageDataset(val_df, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))


Train batches: 2123
Val batches  : 531


In [ ]:
# ============================================================
# 8. Define Model D in PyTorch
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = train_df["label_id"].nunique()

# ============================================================
# 8. Define Model D in PyTorch: ResNet50 frozen
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = train_df["label_id"].nunique()

# Load pretrained ResNet50
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

# Freeze convolutional backbone
for param in model.parameters():
    param.requires_grad = False

# Replace final classification layer
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)

# Make sure new head is trainable
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

print("Device:", device)
print("NUM_CLASSES:", NUM_CLASSES)
print(model.fc)

# Loss, optimizer, scheduler


LR = 1e-4
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1,
    verbose=True
)

print("criterion and optimizer created")

In [17]:
images, labels = next(iter(train_loader))
print(images.shape, labels.shape, images.dtype, labels.dtype)

images, labels = next(iter(val_loader))
print(images.shape, labels.shape, images.dtype, labels.dtype)

torch.Size([32, 3, 224, 224]) torch.Size([32]) torch.float32 torch.int64
torch.Size([32, 3, 224, 224]) torch.Size([32]) torch.float32 torch.int64


In [19]:

# ============================================================
# 9. Checkpoint + resume setup
# ============================================================
BEST_MODEL_PATH = MODEL_ROOT / "best_model.pt"
RESUME_CHECKPOINT_PATH = MODEL_ROOT / "resume_checkpoint.pt"
HISTORY_PATH = OUTPUT_ROOT / "training_history.csv"
METRICS_PATH = OUTPUT_ROOT / "metrics.csv"
PRED_PATH = OUTPUT_ROOT / "val_predictions.csv"
PROBA_PATH = OUTPUT_ROOT / "y_proba.npy"
REPORT_PATH = OUTPUT_ROOT / "classification_report.csv"
META_PATH = OUTPUT_ROOT / "run_metadata.json"

start_epoch = 1
history = []
best_macro_f1 = -np.inf
best_epoch = -1
epochs_without_improvement = 0

def save_full_checkpoint(path, epoch, model, optimizer, scheduler, history, best_macro_f1, best_epoch, epochs_without_improvement):
    payload = {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": history,
        "best_macro_f1": float(best_macro_f1),
        "best_epoch": int(best_epoch),
        "epochs_without_improvement": int(epochs_without_improvement),
        "label2id": label2id,
        "config": {
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "max_epochs": MAX_EPOCHS,
            "model_name": MODEL_NAME,
        },
    }
    torch.save(payload, path)

if RESUME_CHECKPOINT_PATH.exists():
    print("Resume checkpoint found:", RESUME_CHECKPOINT_PATH)
    ckpt = torch.load(RESUME_CHECKPOINT_PATH, map_location=device)

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    history = ckpt.get("history", [])
    best_macro_f1 = ckpt.get("best_macro_f1", -np.inf)
    best_epoch = ckpt.get("best_epoch", -1)
    epochs_without_improvement = ckpt.get("epochs_without_improvement", 0)
    start_epoch = ckpt["epoch"] + 1

    print(f"Resuming from epoch {start_epoch}")
    print(f"Best val macro F1 so far: {best_macro_f1:.6f}")
else:
    print("No resume checkpoint found. Starting fresh.")


No resume checkpoint found. Starting fresh.


In [20]:

# ============================================================
# 10. Training helpers
# ============================================================
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    all_preds = []
    all_true = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_true, all_preds)
    epoch_macro_f1 = f1_score(all_true, all_preds, average="macro")
    epoch_weighted_f1 = f1_score(all_true, all_preds, average="weighted")

    return (
        epoch_loss,
        epoch_acc,
        epoch_macro_f1,
        epoch_weighted_f1,
        np.array(all_true),
        np.array(all_preds),
    )

def predict_with_proba(model, loader):
    model.eval()
    all_probs = []
    all_preds = []
    all_true = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_true.append(labels.cpu().numpy())

    y_proba = np.concatenate(all_probs)
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_true)

    return y_true, y_pred, y_proba

def plot_history(history_df, fig_dir):
    # Loss
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{MODEL_NAME} - Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(fig_dir / "training_loss.png", dpi=200, bbox_inches="tight")
    plt.show()

    # Accuracy
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_acc"], label="Train Accuracy")
    plt.plot(history_df["epoch"], history_df["val_acc"], label="Val Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{MODEL_NAME} - Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(fig_dir / "training_accuracy.png", dpi=200, bbox_inches="tight")
    plt.show()

    # Macro F1
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_macro_f1"], label="Train Macro F1")
    plt.plot(history_df["epoch"], history_df["val_macro_f1"], label="Val Macro F1")
    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.title(f"{MODEL_NAME} - Macro F1")
    plt.legend()
    plt.tight_layout()
    plt.savefig(fig_dir / "training_macro_f1.png", dpi=200, bbox_inches="tight")
    plt.show()


In [21]:
# ============================================================
# 11. Train / resume training
# ============================================================
start_time = time.time()

if start_epoch > MAX_EPOCHS:
    print(f"Training already completed up to epoch {start_epoch - 1}. Skipping training loop.")
else:
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        train_loss, train_acc, train_macro_f1, train_weighted_f1, _, _ = run_epoch(
            model, train_loader, criterion, optimizer=optimizer
        )

        val_loss, val_acc, val_macro_f1, val_weighted_f1, _, _ = run_epoch(
            model, val_loader, criterion, optimizer=None
        )

        scheduler.step(val_macro_f1)
        current_lr = optimizer.param_groups[0]["lr"]

        history.append({
            "epoch": int(epoch),
            "train_loss": float(train_loss),
            "train_acc": float(train_acc),
            "train_macro_f1": float(train_macro_f1),
            "train_weighted_f1": float(train_weighted_f1),
            "val_loss": float(val_loss),
            "val_acc": float(val_acc),
            "val_macro_f1": float(val_macro_f1),
            "val_weighted_f1": float(val_weighted_f1),
            "lr": float(current_lr),
        })

        print(
            f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | "
            f"train_macro_f1={train_macro_f1:.4f} | val_macro_f1={val_macro_f1:.4f} | "
            f"lr={current_lr:.6f}"
        )

        if val_macro_f1 > best_macro_f1:
            best_macro_f1 = val_macro_f1
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"  -> New best model saved at epoch {epoch} (val_macro_f1={val_macro_f1:.4f})")
        else:
            epochs_without_improvement += 1
            print(f"  -> No improvement for {epochs_without_improvement} epoch(s)")

        save_full_checkpoint(
            path=RESUME_CHECKPOINT_PATH,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            history=history,
            best_macro_f1=best_macro_f1,
            best_epoch=best_epoch,
            epochs_without_improvement=epochs_without_improvement,
        )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping triggered after epoch {epoch}")
            break

total_time = time.time() - start_time
print(f"\nElapsed notebook training time: {total_time / 60:.2f} minutes")
print(f"Best epoch: {best_epoch}")
print(f"Best val macro F1: {best_macro_f1:.6f}")


NameError: name 'criterion' is not defined

In [ ]:
# ============================================================
# 11.1 PERMANENT SAVE BLOCK — run immediately after training
# ============================================================
from pathlib import Path
import shutil
import os

# 1) your training output folder
OUT_DIR = Path("/kaggle/working/working_outputs/model_d")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 2) create a small marker file so you can verify this block ran
(OUT_DIR / "SAVE_SUCCESS.txt").write_text(
    "Training finished and save block executed.\n",
    encoding="utf-8"
)

# 3) zip the whole output directory
ZIP_PATH = Path("/kaggle/working/model_d_outputs.zip")
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(
    base_name=str(ZIP_PATH).replace(".zip", ""),
    format="zip",
    root_dir=str(OUT_DIR)
)

# 4) also copy the zip into /kaggle/working/output for visibility if desired
PERSIST_DIR = Path("/kaggle/working/final_artifacts")
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(ZIP_PATH, PERSIST_DIR / ZIP_PATH.name)

# 5) print proof
print("OUT_DIR exists:", OUT_DIR.exists())
print("ZIP exists:", ZIP_PATH.exists())
if ZIP_PATH.exists():
    print("ZIP size MB:", round(ZIP_PATH.stat().st_size / (1024 * 1024), 2))

print("\nContents of OUT_DIR:")
for p in sorted(OUT_DIR.rglob("*")):
    print(" -", p)

print("\nContents of /kaggle/working:")
for p in sorted(Path("/kaggle/working").iterdir()):
    print(" -", p)

In [ ]:

# ============================================================
# 12. Load best model and evaluate on validation split
# ============================================================
if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Best model checkpoint not found: {BEST_MODEL_PATH}")

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))

y_true, y_pred, y_proba = predict_with_proba(model, val_loader)

final_accuracy = accuracy_score(y_true, y_pred)
final_macro_f1 = f1_score(y_true, y_pred, average="macro")
final_weighted_f1 = f1_score(y_true, y_pred, average="weighted")

metrics_df = pd.DataFrame([{
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "augmentation": True,
    "architecture": "EfficientNetB0_Pretrained_Frozen",
    "initial_lr": 1e-4,
    "epochs_trained": len(history),
    "best_epoch": best_epoch,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "weight_decay": 1e-4,
    "scheduler": "ReduceLROnPlateau",
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "accuracy": final_accuracy,
    "macro_f1": final_macro_f1,
    "weighted_f1": final_weighted_f1,
    "training_minutes_this_session": total_time / 60,
}])

print(metrics_df)


In [ ]:

# ============================================================
# 14. Plots
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# build history_df safely from whatever exists
history_df = pd.DataFrame()

if "history" in globals() and hasattr(history, "history"):
    history_df = pd.DataFrame(history.history)
elif "model_history" in globals() and hasattr(model_history, "history"):
    history_df = pd.DataFrame(model_history.history)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
plt.imshow(cm, interpolation="nearest")
plt.title(f"{MODEL_NAME} - Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.savefig(FIG_ROOT / "confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

if len(history_df) > 0:
    plot_history(history_df, FIG_ROOT)
else:
    print("No training history variable found, skipping history plot.")

print("Figure directory:", FIG_ROOT)